In [1]:
!pip install -qU langchain-classic langchain-community langchain-core langchain-groq chromadb sentence-transformers pypdf langchain-text-splitters langchain-openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/7

In [2]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from google.colab import files

/tmp/ipykernel_517/625623912.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
uploaded = files.upload()
pdf_name = list(uploaded.keys())[0]
print('Uploaded:', pdf_name)

Saving Agentic_AI_Complete_Textbook.pdf to Agentic_AI_Complete_Textbook.pdf
Uploaded: Agentic_AI_Complete_Textbook.pdf


In [4]:
pages = PyPDFLoader(pdf_name).load()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(pages)
print('Number of chunks:', len(chunks))

Number of chunks: 661


In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2')
db = Chroma.from_documents(chunks, embeddings)
print('Vector database ready!')

/tmp/ipykernel_517/2042805896.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector database ready!


In [6]:
llm = ChatGroq(api_key="", # enter your api key from groq
               model="llama-3.1-8b-instant", temperature=0)

prompt = ChatPromptTemplate.from_template("Answer based on context: {context}\n\nQuestion: {input}")
combine_docs_chain = create_stuff_documents_chain(llm, prompt)
qa_chain = create_retrieval_chain(db.as_retriever(), combine_docs_chain)

response = qa_chain.invoke({"input": "What is this document about?"})
print(response["answer"])

Based on the provided context, I will design an end-to-end architecture for a document assistant that can answer questions from both unstructured PDFs and a structured SQL database, including the grading layer.

**End-to-End Architecture:**

1. **Input Layer:**
	* Unstructured PDFs: Use `SimpleDirectoryReader` from `llama_index.core` to load and chunk the PDFs automatically.
	* Structured SQL Database: Use a SQLite database to store and query structured data.
2. **Query Routing and Retrieval:**
	* Use a lightweight LLM Classifier (e.g., `llama_index.core.LLMClassifier`) to classify the input query into one of the following categories:
		+ Unstructured PDF
		+ Structured SQL Database
		+ Both (hybrid query)
	* Based on the classification result, route the query to the corresponding retrieval module:
		+ Unstructured PDF: Use the `VectorStoreIndex` from `llama_index.core` to retrieve relevant documents.
		+ Structured SQL Database: Use a SQLite database query to retrieve relevant data.
	